# Classificazione qualità vino

Si leggono i file `wine_train.xlsx` e `wine_test.xlsx`, si controllano i valori mancanti, si bilancia il training set tramite oversampling, si normalizzano le feature, si addestra il modello e si valutano le prestazioni su cross validation e test set.

## Import delle librerie

In questa cella vengono importate le librerie necessarie per:

- gestione dei dati;
- grafici;
- normalizzazione;
- modello di regressione logistica;
- cross validation;
- metriche di valutazione.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## Lettura dei file Excel

Vengono caricati i dataset di training e test dai file Excel.


In [ ]:
train = pd.read_excel("wine_train.xlsx")
test = pd.read_excel("wine_test.xlsx")

print("TRAIN:")
print(train.head())

print("\nTEST:")
print(test.head())

print("\nDimensione train:", train.shape)
print("Dimensione test:", test.shape)


## Controllo dei valori mancanti

Si verifica la presenza di valori mancanti sia nel training set sia nel test set.

In [ ]:
print("\nValori mancanti nel train:")
print(train.isnull().sum())

print("\nValori mancanti nel test:")
print(test.isnull().sum())

for col in train.select_dtypes(include=np.number).columns:
    train[col] = train[col].fillna(train[col].mean())

for col in test.select_dtypes(include=np.number).columns:
    test[col] = test[col].fillna(test[col].mean())


## Separazione tra feature e target

La colonna `WineQuality` viene usata come variabile target.

Tutte le altre colonne vengono usate come feature del modello.

In [ ]:
X_train = train.drop(columns=["WineQuality"])
y_train = train["WineQuality"]

X_test = test.drop(columns=["WineQuality"])
y_test = test["WineQuality"]

print("\nFeature usate:")
print(X_train.columns)

print("\nDistribuzione classi nel train:")
print(y_train.value_counts().sort_index())


## Istogramma della variabile Alcohol

Questo istogramma mostra la distribuzione della variabile `alcohol` nel dataset di training originale.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(train["alcohol"], bins=20)
plt.title("Distribuzione della variabile Alcohol")
plt.xlabel("Alcohol")
plt.ylabel("Frequenza")
plt.show()


## Scatter plot Alcohol vs Density

Il grafico a dispersione permette di osservare la relazione tra la variabile `alcohol` e la variabile `density`.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(train["alcohol"], train["density"], alpha=0.5)
plt.title("Relazione tra Alcohol e Density")
plt.xlabel("Alcohol")
plt.ylabel("Density")
plt.show()


## Normalizzazione

Le feature vengono normalizzate con `MinMaxScaler`, che porta i valori numerici in un intervallo compreso tra 0 e 1.

Lo scaler viene adattato solo sul training set e poi applicato anche al test set.

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("\nDataset normalizzato:")
print(X_train_scaled.head())


## Modello: Regressione Logistica

Viene creato un modello di regressione logistica.

Il parametro `max_iter=5000` aumenta il numero massimo di iterazioni, riducendo il rischio che l'algoritmo non converga.

In [ ]:
model = LogisticRegression(
    max_iter=5000,
    solver="lbfgs"
)


## Cross Validation

La cross validation viene eseguita con `StratifiedKFold`, che mantiene la proporzione delle classi nei diversi fold.

Sono usati 3 fold perché alcune classi hanno pochi esempi nel training set originale.

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("\nRisultati Cross Validation:")
print("Accuracy per ogni fold:", scores)
print("Accuracy media:", scores.mean())
print("Deviazione standard:", scores.std())


## Grafico dell'accuracy per fold

Il grafico mostra l'accuracy ottenuta in ciascun fold della cross validation.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(scores, marker="o")
plt.title("Accuracy per fold - Cross Validation")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.xticks(range(len(scores)), ["Fold 1", "Fold 2", "Fold 3"])
plt.show()


## Predizioni in cross validation

Con `cross_val_predict` vengono generate predizioni sul training set seguendo la logica della cross validation.

Queste predizioni permettono di calcolare matrice di confusione e classification report sulla cross validation.

In [ ]:
y_train_pred_cv = cross_val_predict(
    model,
    X_train_scaled,
    y_train,
    cv=cv
)

print("\nMatrice di confusione - Cross Validation:")
print(confusion_matrix(y_train, y_train_pred_cv))

print("\nClassification report - Cross Validation:")
print(classification_report(y_train, y_train_pred_cv, zero_division=0))


## Training finale su tutto il training set

Dopo la cross validation, il modello viene addestrato usando tutto il training set bilanciato e normalizzato.

In [ ]:
model.fit(X_train_scaled, y_train)


## Test finale

Il modello addestrato viene valutato sul test set.

Vengono calcolati accuracy, matrice di confusione e classification report.

In [ ]:
y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy sul test set:", accuracy)

print("\nMatrice di confusione - Test:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report - Test:")
print(classification_report(y_test, y_pred, zero_division=0))


## Grafico della matrice di confusione

La matrice di confusione viene visualizzata graficamente.

Le righe rappresentano le classi reali, mentre le colonne rappresentano le classi predette dal modello.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("Matrice di confusione - Regressione Logistica")
plt.xlabel("Classe predetta")
plt.ylabel("Classe reale")
plt.colorbar()

classi = sorted(y_train.unique())
plt.xticks(range(len(classi)), classi)
plt.yticks(range(len(classi)), classi)

for i in range(len(cm)):
    for j in range(len(cm[i])):
        plt.text(j, i, cm[i][j], ha="center", va="center")

plt.show()


# Confronto senza flag

In questa parte viene ripetuto l'addestramento rimuovendo la variabile `Synthetic_Noise_Flag`.

L'obiettivo è confrontare le prestazioni del modello con e senza questa feature.

## Preparazione dei dati senza `Synthetic_Noise_Flag`

La colonna `Synthetic_Noise_Flag` viene rimossa sia dal training set sia dal test set.

In [ ]:
X_train_nf = train.drop(columns=["WineQuality", "Synthetic_Noise_Flag"], errors="ignore")
y_train_nf = train["WineQuality"]

X_test_nf = test.drop(columns=["WineQuality", "Synthetic_Noise_Flag"], errors="ignore")
y_test_nf = test["WineQuality"]


## Normalizzazione senza flag

Anche in questo confronto le feature vengono normalizzate con `MinMaxScaler`.

In [ ]:
scaler_nf = MinMaxScaler()
X_train_nf_scaled = scaler_nf.fit_transform(X_train_nf)
X_test_nf_scaled = scaler_nf.transform(X_test_nf)


## Modello e cross validation senza flag

Viene addestrato un secondo modello di regressione logistica senza usare la variabile `Synthetic_Noise_Flag`.

Il risultato della cross validation viene confrontato con quello del modello precedente.

In [ ]:
model_nf = LogisticRegression(max_iter=5000)

scores_nf = cross_val_score(
    model_nf,
    X_train_nf_scaled,
    y_train_nf,
    cv=cv,
    scoring="accuracy"
)

print("\n=== SENZA FLAG ===")
print("Accuracy CV media:", scores_nf.mean())


## Test finale senza flag

Il secondo modello viene addestrato su tutto il training set senza flag e poi valutato sul test set.

In [ ]:
model_nf.fit(X_train_nf_scaled, y_train_nf)

y_pred_nf = model_nf.predict(X_test_nf_scaled)
accuracy_nf = accuracy_score(y_test_nf, y_pred_nf)

print("Accuracy test senza flag:", accuracy_nf)
